# PhenoBench YOLO Export (multiclass)

Exports PhenoBench as **YOLO** datasets (images + `labels/*.txt` + `data.yaml`) for WongKinYiu/yolov7 training, in two variants:

- **full** — 1024² frames, copied verbatim.
- **tiled512** — 3×3 tiling with 0.5 overlap (~512 px tiles, matching a 2×2-no-overlap crop while recovering tile-boundary plants).

**Partials (do-not-care):** YOLO has no do-not-care flag, so partial (`plant_visibility <= 0.5`) plants are **dropped** from the label files (`include_partials=False`). The official PhenoBench evaluator re-derives partial handling straight from the masks at eval time, so this does not affect leaderboard comparability.

## Environment

In [1]:
!python --version

Python 3.10.10


In [2]:
!pip install -q --no-cache-dir phenobench
!pip install -q --no-cache-dir --no-deps git+https://github.com/frdiener/agri-vision-edge.git@abf74003b804d5c7130e368e0ae0eef696fa047e

## Configuration

In [3]:
from pathlib import Path

RAW_ROOT = Path("/kaggle/input/datasets/freimutdiener/phenobench-raw-dataset-v1-1-0/PhenoBench")
WORK = Path("/kaggle/working")
SPLITS = ["train", "val"]

# Upstream do-not-care criterion (plant_visibility <= ratio). Partials are
# dropped from the YOLO labels; the upstream evaluator handles them at eval.
PARTIAL_THRESHOLD = 0.5
MIN_BOX_SIZE = 0.0

# variant -> materialization recipe
VARIANTS = {
    "full": dict(tiling=None),
    "tiled512": dict(tiling=(3, 3, 0.5)),   # rows, cols, overlap -> ~512 px
}

## Export

In [4]:
import json
from phenobench import PhenoBench

from agri_vision_edge.data import (
    PHENOBENCH_MULTICLASS,
    export_yolo_split,
    write_data_yaml,
)
from agri_vision_edge.data.plant_boxes import PartialAwarePhenoBench
from agri_vision_edge.data.tiling import TiledPhenoBench, FilterConfig

DATASET_DEF = PHENOBENCH_MULTICLASS


def build_split(split, recipe):
    tiling = recipe["tiling"]
    if tiling is None:
        # Partial-aware full frames: PartialAwarePhenoBench tags is_partial
        # from plant_visibility so export_yolo_split can drop them.
        base = PhenoBench(
            root=str(RAW_ROOT),
            split=split,
            target_types=["semantics", "plant_instances", "plant_visibility"],
            ignore_partial=False,
        )
        ds = PartialAwarePhenoBench(base, partial_threshold=PARTIAL_THRESHOLD)
        # full images already exist on disk -> copy, don't re-encode
        return ds, RAW_ROOT / split / "images"

    rows, cols, overlap = tiling
    source = PhenoBench(
        root=str(RAW_ROOT),
        split=split,
        target_types=["semantics", "plant_instances", "plant_visibility"],
        ignore_partial=False,
    )
    ds = TiledPhenoBench(
        source,
        rows=rows,
        cols=cols,
        overlap=overlap,
        filter_config=FilterConfig(
            min_instance_pixels=200,   # drop tiny tile-border fragments
        ),
        partial_threshold=PARTIAL_THRESHOLD,
    )
    return ds, None  # tiles are synthesised -> re-encode


manifest = {"dataset": "phenobench_multiclass", "variants": {}}

for variant, recipe in VARIANTS.items():
    out_dir = WORK / variant
    summaries = {}
    for split in SPLITS:
        ds, source_images_dir = build_split(split, recipe)
        summaries[split] = export_yolo_split(
            ds, DATASET_DEF, out_dir, split,
            min_box_size=MIN_BOX_SIZE,
            source_images_dir=source_images_dir,
            include_partials=False,   # drop do-not-care partials
        )
        print(variant, summaries[split])
    write_data_yaml(out_dir, DATASET_DEF, train_split="train", val_split="val")
    manifest["variants"][variant] = {"recipe": {k: str(recipe[k]) for k in recipe},
                                     "splits": summaries}

(WORK / "export_manifest.json").write_text(json.dumps(manifest, indent=2))
print("\nwrote", WORK / "export_manifest.json")

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


full {'split': 'train', 'images': 1407, 'boxes': 16450, 'skipped_boxes': 3566, 'images_dir': '/kaggle/working/full/images/train', 'labels_dir': '/kaggle/working/full/labels/train'}
full {'split': 'val', 'images': 772, 'boxes': 8568, 'skipped_boxes': 1840, 'images_dir': '/kaggle/working/full/images/val', 'labels_dir': '/kaggle/working/full/labels/val'}
tiled512 {'split': 'train', 'images': 12663, 'boxes': 37388, 'skipped_boxes': 4846, 'images_dir': '/kaggle/working/tiled512/images/train', 'labels_dir': '/kaggle/working/tiled512/labels/train'}
tiled512 {'split': 'val', 'images': 6948, 'boxes': 19725, 'skipped_boxes': 2570, 'images_dir': '/kaggle/working/tiled512/images/val', 'labels_dir': '/kaggle/working/tiled512/labels/val'}

wrote /kaggle/working/export_manifest.json


## Verify

In [5]:
!find /kaggle/working -maxdepth 2 -type d | sort
!echo '--- full/data.yaml ---'; cat /kaggle/working/full/data.yaml
!echo '--- tiled512/data.yaml ---'; cat /kaggle/working/tiled512/data.yaml
!du -sh /kaggle/working/full /kaggle/working/tiled512

/kaggle/working
/kaggle/working/full
/kaggle/working/full/images
/kaggle/working/full/labels
/kaggle/working/tiled512
/kaggle/working/tiled512/images
/kaggle/working/tiled512/labels
--- full/data.yaml ---
train: /kaggle/working/full/images/train
val: /kaggle/working/full/images/val
nc: 2
names: ['crop', 'weed']
--- tiled512/data.yaml ---
train: /kaggle/working/tiled512/images/train
val: /kaggle/working/tiled512/images/val
nc: 2
names: ['crop', 'weed']
5.3G	/kaggle/working/full
12G	/kaggle/working/tiled512
